In [ ]:
import os
import openai
import yaml
import zipfile
from functools import lru_cache
import uuid

In [ ]:
knwf_path = "test_data/2024_nuclei_segmentation_knime.knwf"
extract_dir = "test_data_unzipped"

1. Extract a KNIME .knwf file into the target directory.

In [ ]:
# Access zip file content directly, Convert it to bytestream directly
# Comments in english
def unzip_knime_workflow(zip_path: str, extract_to: str):
    """
    Entpackt eine KNIME .knwf Datei in das angegebene Zielverzeichnis.
    """
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)
    print(f"Entpackt: {zip_path} → {extract_to}")

unzip_knime_workflow(knwf_path, extract_dir)

2. Workflow-Verzeichnis finden

In [ ]:
def find_workflow_dir(base_dir: str) -> str:
    """
    Sucht im Verzeichnis base_dir nach dem ersten Unterordner, der eine Datei namens 'workflow.knime' enthält.
    Gibt dann den vollständigen Pfad zu diesem Unterordner zurück.
    """
    for entry in os.listdir(base_dir):  # Alle Dateien und Ordner in base_dir durchgehen
        full_path = os.path.join(base_dir, entry)

        # Wenn es ein Ordner ist und darin 'workflow.knime' liegt → Erfolg
        if os.path.isdir(full_path) and "workflow.knime" in os.listdir(full_path):
            print(f"Workflow-Ordner gefunden: {full_path}")
            return full_path

    # Wenn kein solcher Ordner gefunden wurde → Fehler auslösen
    raise ValueError(f"⚠️ Kein gültiger Workflow-Ordner gefunden in {base_dir}")

workflow_dir = find_workflow_dir(extract_dir)


3. Sammelt alle settings.xml-Dateien aus den Node-Unterordnern und gibt sie als dict zurück.
    Key: Node-Name (z. B. "Image Reader (#1)")
    Value: XML-String

In [ ]:
def collect_knime_node_files(workflow_dir: str):
    """
    Sammelt alle settings.xml-Dateien aus den Node-Unterordnern und gibt sie als dict zurück.
    Key: Node-Name (z. B. "Image Reader (#1)")
    Value: XML-String
    """
    node_data = {}

    for entry in os.listdir(workflow_dir):
        full_path = os.path.join(workflow_dir, entry)
        if os.path.isdir(full_path) and not entry.startswith('.'):
            xml_path = os.path.join(full_path, "settings.xml")
            if os.path.exists(xml_path):
                with open(xml_path, 'r', encoding='utf-8') as f:
                    xml_content = f.read()
                    node_data[entry] = xml_content

    return node_data

In [ ]:
knime_nodes = collect_knime_node_files(workflow_dir)
knime_nodes

In [ ]:
# Collects the workflow.knime file
def collect_workflow_file(workflow_dir: str) -> str:
    """
    Sammelt den Inhalt der workflow.knime-Datei und gibt ihn als String zurück.
    """
    workflow_path = os.path.join(workflow_dir, "workflow.knime")
    if not os.path.exists(workflow_path):
        raise FileNotFoundError(f"Die Datei {workflow_path} wurde nicht gefunden.")
    
    with open(workflow_path, 'r', encoding='utf-8') as f:
        return f.read()


In [ ]:
workflow_content = collect_workflow_file(workflow_dir)
print(workflow_content[:500])  # Ausgabe der ersten 500 Zeichen des Workflows

In [ ]:
knime_nodes = collect_knime_node_files(workflow_dir)
print("Erste 100 Zeichen jeder Node:")
for node_name, xml_content in knime_nodes.items():
    print(f"{node_name}: {xml_content[:100]}...")

In [ ]:
@lru_cache(maxsize=1)
def load_translation_examples(yaml_path: str = "data/translation_table.yml") -> list:
    with open(yaml_path, "r", encoding="utf-8") as f:
        docs = list(yaml.safe_load_all(f))
        print(docs)
        examples = []
        for doc in docs[0]:
            knime = doc.get("KNIME")
            galaxy = doc.get("Galaxy")

            if knime and galaxy:
                examples.append({
                    "KNIME": knime.strip(),
                    "Galaxy": galaxy.strip()
                })

    return examples


In [ ]:
translation_examples = load_translation_examples()
translation_examples

In [ ]:
def build_examples():
    examples_text =  """You are a translator that converts KNIME workflow nodes to Galaxy workflow steps.

Below are examples of how this translation should be done:
"""
    examples = load_translation_examples()
    if examples:
        for i, ex in enumerate(examples[:6]):  # z. B. nur 6 Beispiele
            examples_text += f"""

## Example {i + 1}:

KNIME node (XML):

```xml
{ex["KNIME"]}
```

Galaxy step (JSON):
```json
{ex["Galaxy"]}
```

"""
    return examples_text


In [ ]:
examples = build_examples()
print(examples)

In [ ]:
# Dict-Inhalt in String umwandeln
knime_nodes_str = "\n".join(
    f"Node ID: {key}\n{value}" for key, value in knime_nodes.items()
)
print(knime_nodes_str)

In [ ]:
def build_task_prompt():
    
    task_prompt = f"""
# Your Task
You are a system that translates complete KNIME workflows into Galaxy workflows. Produce a **single, valid Galaxy .ga workflow JSON** that can be imported in Galaxy, representing the entire KNIME workflow below.

# Input
KNIME Nodes (XML):
```xml
{knime_nodes_str}
```

The KNIME workflow content (XML):
```xml
{workflow_content}
```

# Output Requirements
Respond with the complete Galaxy workflow JSON object ONLY (no markdown fences, no comments, no explanations).

The JSON must be a valid Galaxy .ga workflow 
Do not include TODOs or comments in the JSON.

"""
    
    return task_prompt

In [ ]:
task = build_task_prompt()
print(task)

In [ ]:
full_prompt  = f"{examples}\n\n{task}"
print(full_prompt)

In [ ]:
def list_scadsai_models():
    client = openai.OpenAI(
        base_url="https://llm.scads.ai/v1",
        api_key=os.environ.get("SCADSAI_API_KEY")
    )
    models = client.models.list()
    return [m.id for m in models.data]

In [ ]:
list_scadsai_models()

In [ ]:
def prompt_scadsai_llm(message:str, model="openai/gpt-oss-120b"):
    """A prompt helper function that sends a message to ScaDS.AI LLM server at 
    ZIH TU Dresden and returns only the text response.
    """
    
    # convert message in the right format if necessary
    if isinstance(message, str):
        message = [{"role": "user", "content": message}]
    
    # setup connection to the LLM
    client = openai.OpenAI(base_url="https://llm.scads.ai/v1",
                           api_key=os.environ.get('SCADSAI_API_KEY')
    )
    response = client.chat.completions.create(
        model=model,
        messages=message
    )
    
    # extract answer
    return response.choices[0].message.content

In [ ]:
answer = prompt_scadsai_llm(message= full_prompt)
print(answer)

In [ ]:
# Parse the answer to a JSON object

In [ ]:
# Go through the JSON object recursively searching for the "uuid" key and replace the correspondign value with a new UUID
# Use the uuid module to generate a new UUID

In [ ]:
# Save the answer to a file
output_file = "knime2galaxy_workflow_output.ga"
with open(output_file, "w", encoding="utf-8") as f:
    f.write(answer)

print(f"Galaxy workflow saved to {output_file}")

In [ ]:
i = 0
while i < 10:
    i += 1
    myuuid = uuid.uuid4()
    print(f"Generated UUID: {myuuid}")



In [ ]:
# Compare generated Galaxy workflow with the original one
# implement uuids
# use planemo create workflows and find errors
# KNIME Jupyter Notebook
# Galaxy Jupyter Notebook and Compare the two


